In [1]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH = 'duy'

# Install lightweight dependencies needed for smoke checks.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'torch', '-q'],
    check=True,
)

# Clone or update exact branch used for current development.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', os.getcwd())
print('Branch:', BRANCH)

Repo ready: /content/Yolo-ST-GCN
Branch: duy


# ST-GCN Smoke Test
Notebook này chạy smoke test nhanh cho pipeline sau khi tách Penn/COCO dataset.

Mục tiêu:
- Xác nhận import module thành công
- Xác nhận forward pass của model
- Xác nhận build tensors cho Penn và COCO synthetic đều trả về định dạng chung `(N, 2, T, 14, 1)`

In [2]:
import os
import sys
import subprocess
from pathlib import Path


def find_repo_root() -> Path | None:
    """Try common Colab/local locations for this repo."""
    candidates = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]

    for base in candidates:
        if (base / 'src').exists() and (base / 'scripts').exists():
            return base

    markers = ['Yolo-ST-GCN', 'ST-GCN']
    for base in candidates:
        if not base.exists():
            continue
        for m in markers:
            found = list(base.glob(f'**/{m}'))[:5]
            for p in found:
                if (p / 'src').exists() and (p / 'scripts').exists():
                    return p
    return None


def ensure_repo_available() -> Path | None:
    repo = find_repo_root()
    if repo is not None:
        return repo

    # Colab fallback: clone repo if missing.
    colab_repo = Path('/content/Yolo-ST-GCN')
    if not colab_repo.exists():
        print('Repo not found in kernel filesystem. Cloning from GitHub...')
        subprocess.run(
            ['git', 'clone', 'https://github.com/schizocatto/Yolo-ST-GCN.git', str(colab_repo)],
            check=True,
        )
    if (colab_repo / 'src').exists() and (colab_repo / 'scripts').exists():
        return colab_repo

    return None


repo_root = ensure_repo_available()
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Python:', sys.version.split()[0])
print('CWD:', Path.cwd())
print('Repo root detected:', repo_root)
print('src import ready:', repo_root is not None and (repo_root / 'src').exists())

Python: 3.12.13
CWD: /content/Yolo-ST-GCN
Repo root detected: /content/Yolo-ST-GCN
src import ready: True


In [3]:
import inspect
import os
import tempfile
import numpy as np
import scipy.io
import torch

from src.config import TARGET_FRAMES, EXERCISE_CLASSES
from src.model import Model_STGCN
from src.dataset import build_data_tensors

has_dataset_format = 'dataset_format' in inspect.signature(build_data_tensors).parameters
print('build_data_tensors supports dataset_format:', has_dataset_format)

# 1) Model forward smoke
x = np.random.randn(2, 2, TARGET_FRAMES, 14, 1).astype(np.float32)
model = Model_STGCN(num_classes=len(EXERCISE_CLASSES))
out = model(torch.from_numpy(x))
print('Model forward output shape:', tuple(out.shape))

# 2) Penn synthetic dataset smoke
with tempfile.TemporaryDirectory(prefix='penn_smoke_') as td:
    action = EXERCISE_CLASSES[0]
    n_frames = 48
    mat = {
        'x': np.random.randint(0, 640, (n_frames, 13)).astype(float),
        'y': np.random.randint(0, 480, (n_frames, 13)).astype(float),
        'action': np.array([[action]]),
        'nframes': np.array([[n_frames]]),
        'train': np.array([[1]]),
    }
    scipy.io.savemat(os.path.join(td, '0001.mat'), mat)
    if has_dataset_format:
        data_p, labels_p, flags_p, _, vids_p = build_data_tensors(td, dataset_format='penn')
    else:
        data_p, labels_p, flags_p, _, vids_p = build_data_tensors(td)
    print('Penn tensor shape:', data_p.shape)
    print('Penn labels:', labels_p.tolist(), 'flags:', flags_p.tolist(), 'video_ids:', vids_p)

assert out.shape[0] == 2 and out.shape[1] == len(EXERCISE_CLASSES)
assert data_p.shape[1:] == (2, TARGET_FRAMES, 14, 1)

# 3) COCO synthetic dataset smoke (only when new API is available)
if has_dataset_format:
    with tempfile.TemporaryDirectory(prefix='coco_smoke_') as td:
        action = EXERCISE_CLASSES[1]
        kpts = np.random.rand(40, 17, 2).astype(np.float32) * np.array([640.0, 480.0], dtype=np.float32)
        np.savez(
            os.path.join(td, 'sample_0001.npz'),
            keypoints=kpts,
            action=np.array([action]),
            train=np.array([1]),
            video_id=np.array(['sample_0001'])
        )
        data_c, labels_c, flags_c, _, vids_c = build_data_tensors(td, dataset_format='coco')
        print('COCO->Penn tensor shape:', data_c.shape)
        print('COCO labels:', labels_c.tolist(), 'flags:', flags_c.tolist(), 'video_ids:', vids_c)
        assert data_c.shape[1:] == (2, TARGET_FRAMES, 14, 1)
    print('SMOKE TEST: PASS (Penn + COCO)')
else:
    print('SMOKE TEST: PASS (Penn only)')
    print('Note: Repo branch on GitHub chua co API dataset_format; can push latest code de test COCO.')

build_data_tensors supports dataset_format: True
Model forward output shape: (2, 8)
[dataset] No official test split found for Penn data; using last 30% per class as test.
Penn tensor shape: (1, 2, 64, 14, 1)
Penn labels: [0] flags: [0] video_ids: ['0001']
[dataset] No test split found for COCO data; using last 30% per class as test.
COCO->Penn tensor shape: (1, 2, 64, 14, 1)
COCO labels: [1] flags: [0] video_ids: ['sample_0001']
SMOKE TEST: PASS (Penn + COCO)


/content/Yolo-ST-GCN/src/graph.py:68: RuntimeWarning: divide by zero encountered in divide
  D_inv   = np.where(row_sum > 0, 1.0 / row_sum, 0.0)


## Train + Inference on Gym288-skeleton
Chạy 2 cell bên dưới để huấn luyện và đánh giá ST-GCN trên Gym288-skeleton.

Yêu cầu dữ liệu:
- File pickle theo format docs: `gym288_skeleton.pkl`
- Có keys `split.train`, `split.test`, `annotations`

Kết quả sẽ được lưu trong `outputs/gym288/`.

In [4]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# Thu muc luu dataset Gym288-skeleton trong Colab
DOWNLOAD_DIR = Path('/content/Gym288-skeleton')

if DOWNLOAD_DIR.exists() and any(DOWNLOAD_DIR.iterdir()):
    print('Dataset da ton tai — bo qua buoc tai xuong.')
else:
    print('Dang tai dataset Gym288-skeleton tu Hugging Face...')
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(DOWNLOAD_DIR),
        local_dir_use_symlinks=False,
    )
    print('Da tai xong!')

print('Cau truc thu muc tai ve:')
for item in sorted(os.listdir(DOWNLOAD_DIR)):
    print(f' - {item}')

# Tu dong tim file pickle dau vao cho train/inference
pkl_candidates = sorted(DOWNLOAD_DIR.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError('Khong tim thay file .pkl trong Gym288-skeleton sau khi tai.')

GYM288_DATASET_PATH = pkl_candidates[0]
OUT_DIR = Path('outputs/gym288')

print('Using dataset pickle:', GYM288_DATASET_PATH)
print('Output dir:', OUT_DIR)

Dataset da ton tai — bo qua buoc tai xuong.
Cau truc thu muc tai ve:
 - .cache
 - .gitattributes
 - README.md
 - gym_288_skeleton.pkl
Using dataset pickle: /content/Gym288-skeleton/gym_288_skeleton.pkl
Output dir: outputs/gym288


In [ ]:
import subprocess
import sys

if 'GYM288_DATASET_PATH' not in globals():
    raise RuntimeError('Hay chay Cell 7 truoc de tai dataset va khoi tao GYM288_DATASET_PATH.')

# 1) Train ST-GCN on Gym288 train split
train_cmd = [
    sys.executable, 'scripts/train_gym288.py',
    '--dataset_path', str(GYM288_DATASET_PATH),
    '--out_dir', str(OUT_DIR),
    '--epochs', '30',
    '--batch_size', '32',
    '--lr', '0.001',
]
print('Running train command:\n', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)


Running train command:
 /usr/bin/python3 scripts/train_gym288.py --dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl --out_dir outputs/gym288 --epochs 30 --batch_size 32 --lr 0.001


In [ ]:

# # 2) Inference/Evaluation on Gym288 test split
# weights_path = OUT_DIR / 'stgcn_gym288.pth'
# infer_cmd = [
#     sys.executable, 'scripts/inference_gym288.py',
#     '--dataset_path', str(GYM288_DATASET_PATH),
#     '--weights', str(weights_path),
#     '--out_dir', str(OUT_DIR),
#     '--batch_size', '64',
#     '--topk', '5',
# ]
# print('Running inference command:\n', ' '.join(infer_cmd))
# subprocess.run(infer_cmd, check=True)

# print('\nDone. Check outputs in:', OUT_DIR)
# print('- metrics_train_gym288.json')
# print('- metrics_inference_gym288.json')
# print('- predictions_test_top1.json')